In [ ]:
import numpy as np 
import pickle
import torch
import pandas as pd
from functions import Reduced_space_PDF_function, Codec, Connectivity

In [ ]:
# Define Variables
coordinates  = np.array(pd.read_csv('Dimensionality_reduction\Adjency_matrix\\coordinates.csv', header = None))
point_id = np.argsort(coordinates[:, -1].astype(int))
connectivity = pd.read_csv('Dimensionality_reduction\Adjency_matrix\\adjacency_matrix.csv')
edge_indices = connectivity[['point_i', 'point_j']]
edge_weight  = connectivity[['distance']]
output_dir = 'output\\'

In [5]:
np.random.seed(0)  
dataset_pressure_gradient = np.load('input\\dataset_pressure_gradient.npy') # Sum the variable for each sample (along the first axis)
summed_variables = np.sum(dataset_pressure_gradient, axis=0) # Normalize the summed variables
normalized_pressures = summed_variables / max(summed_variables) # Now we have one sample that indicates the regions with most influence for all of the 70 samples
pressure = np.array(normalized_pressures[:, 0])

In [5]:
### FIRST REDUCTION

# Find Points in Reduced Space
npoints = 28600 
pressure_reduced, coordinates_reduced = Reduced_space_PDF_function.get_reduced_space(pressure, coordinates, point_id, p1=0.2, pn=1.0, num_selected=npoints)
point_id_reduced = coordinates_reduced[:, -1].astype(int)

# Calculate Pooling and Unpooling interpolation coefficients and respective Point_ID
normalized_dataset = pressure_reduced[:,0]
reduced_space = Codec.Codec(normalized_dataset, point_id_reduced, coordinates)
intrp_coeffs_encoder, point_id_data_encoder = reduced_space.reduced_connectivity(mode = 'encoder')
intrp_coeffs_decoder, point_id_data_decoder = reduced_space.reduced_connectivity(mode = 'decoder')

# Write Connectivity Matrix
reduced_space = Connectivity.WriteConnectivity(normalized_dataset, point_id_reduced, coordinates)
reduced_conn = reduced_space.reduced_connectivity()

# Save outputs
torch.save(intrp_coeffs_decoder, f'{output_dir}decoder_interp_coeff_list_{npoints}.pt')
torch.save(point_id_data_decoder, f'{output_dir}decoder_point_id_data_list_{npoints}.pt')
torch.save(intrp_coeffs_encoder, f'{output_dir}encoder_interp_coeff_list_{npoints}.pt')
torch.save(point_id_data_encoder, f'{output_dir}encoder_point_id_data_list_{npoints}.pt')

with open(f'{output_dir}reduced_connectivity_array_{npoints}.pkl', 'wb') as file:
    pickle.dump(reduced_conn, file)

In [6]:
### SECOND REDUCTION

# Find Points in Reduced Space
npoints = 9600
coordinates_reduced[:, -1] = np.argsort(coordinates_reduced[:, -1].astype(int)) # Order the Point_ID from 0 to max value of reduced_space
point_id_reduced = np.argsort(coordinates_reduced[:, -1].astype(int)) 

pressure_reduced_1, coordinates_reduced_1 = Reduced_space_PDF_function.get_reduced_space(pressure_reduced, coordinates_reduced, point_id_reduced, p1=0.2, pn=1.0, num_selected=npoints)
# coordinates_reduced_1[:, -1] = np.argsort(coordinates_reduced_1[:, -1].astype(int)) # Order the Point_ID from 0 to max value of reduced_space

point_id_reduced_1 = coordinates_reduced_1[:, -1].astype(int)

# Calculate Pooling and Unpooling interpolation coefficients and respective Point_ID
normalized_dataset_1 = np.array([el[0] for el in pressure_reduced_1[:,0]])
reduced_space = Codec.Codec(normalized_dataset_1, point_id_reduced_1, coordinates_reduced)
intrp_coeffs_encoder, point_id_data_encoder = reduced_space.reduced_connectivity(mode = 'encoder')
intrp_coeffs_decoder, point_id_data_decoder = reduced_space.reduced_connectivity(mode = 'decoder')

# Write Connectivity Matrix
reduced_space = Connectivity.WriteConnectivity(normalized_dataset_1, point_id_reduced_1, coordinates_reduced)
reduced_conn = reduced_space.reduced_connectivity()

# Save outputs
torch.save(intrp_coeffs_decoder, f'{output_dir}decoder_interp_coeff_list_{npoints}.pt')
torch.save(point_id_data_decoder, f'{output_dir}decoder_point_id_data_list_{npoints}.pt')
torch.save(intrp_coeffs_encoder, f'{output_dir}encoder_interp_coeff_list_{npoints}.pt')
torch.save(point_id_data_encoder, f'{output_dir}encoder_point_id_data_list_{npoints}.pt')

with open(f'{output_dir}reduced_connectivity_array_{npoints}.pkl', 'wb') as file:
    pickle.dump(reduced_conn, file)